# 🌄 Hessian Spectrum Visualization — Curvature, Flat Minima, and Why Conditioning Matters

**What this notebook does**

- Builds a *tiny* PyTorch MLP (2 inputs, a handful of hidden units, 2-class output) on a toy dataset —
  small enough (a couple hundred parameters at most) that its **exact** loss Hessian is a
  few-hundred-by-few-hundred matrix we can form and eigendecompose directly, no approximations.
- Computes the exact Hessian via `torch.autograd.functional.hessian` (double backward under the hood).
- Plots the eigenvalue spectrum and checks it against the well-known qualitative shape reported for real
  (much larger) neural network loss landscapes: a handful of large eigenvalues plus a bulk clustered near
  zero (Sagun et al. 2017; Ghorbani et al. 2019).
- Computes the condition number of the Hessian and connects it directly to why plain GD/SGD step-size
  selection is hard under ill-conditioned curvature — the motivation for the curvature-aware optimizers
  (Natural Gradient, K-FAC, EKFAC) covered elsewhere in this repo.
- Visualizes flat vs. sharp directions by perturbing the trained parameters along the top and bottom
  Hessian eigenvectors and plotting the resulting 1D loss curves.

**Pedagogical notes.** We work in `float64` throughout (numerical precision matters for eigenvalues near
zero) and keep the network deliberately tiny. This is **not** a recipe for computing Hessians of real
(million-plus parameter) networks — exact Hessians are $O(P^2)$ to store and $O(P^3)$ to eigendecompose,
which is precisely *why* K-FAC/EKFAC-style Kronecker approximations exist.

**References**
- Sagun, L., Evci, U., Güney, V. U., Dauphin, Y., & Bottou, L. (2017). *Empirical Analysis of the
  Hessian of Over-Parametrized Neural Networks.*
- Ghorbani, B., Krishnan, S., & Xiao, Y. (2019). *An Investigation into Neural Net Optimization via
  Hessian Eigenvalue Density.*
- Martens, J., & Grosse, R. (2015). *Optimizing Neural Networks with Kronecker-Factored Approximate
  Curvature.* (motivates why Hessian conditioning matters for optimizer design)

## 🔍 Conceptual Intuition

For a loss $L(\theta)$ over $P$ parameters, the Hessian $H = \nabla^2_\theta L$ is a $P \times P$
symmetric matrix describing local curvature. Its eigenvalues $\lambda_1 \geq \dots \geq \lambda_P$ tell
you, for each orthogonal direction $v_i$ in parameter space, how quickly the loss curves along that
direction: $L(\theta^* + t v_i) \approx L(\theta^*) + \tfrac{1}{2}\lambda_i t^2$ near a critical point
$\theta^*$.

Empirically, real (large) neural network loss Hessians near a trained minimum tend to have a very
distinctive spectrum: **a small number of large eigenvalues (sharp directions) and a large bulk of
eigenvalues clustered near zero (flat directions)** — see Sagun et al. (2017) and Ghorbani et al. (2019).
The rough intuition offered in that line of work: the "outlier" eigenvalues track something like the
number of output classes / distinguishable sub-problems, while the near-zero bulk reflects
directions the optimization is (over-)parameterized enough to leave largely unconstrained.

Why this matters for optimizers: a first-order method like GD/SGD uses a single learning rate $\eta$
for every direction simultaneously. Stability along the sharpest direction requires roughly
$\eta \lesssim 2/\lambda_{\max}$, but progress along the flattest direction is governed by
$\eta \cdot \lambda_{\min}$. When $\lambda_{\max}/\lambda_{\min}$ (the **condition number**) is large,
these two requirements pull in opposite directions — no single $\eta$ is good for both, and convergence
along flat directions becomes painfully slow relative to the largest stable step size. Curvature-aware
methods (Newton, Natural Gradient, K-FAC/EKFAC) exist precisely to rescale each direction by
(an approximation to) its own curvature instead of using one global $\eta$.

We will verify this shape and this conditioning story directly on our tiny network, rather than just
citing it.

### Step-1: Imports and Environment Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, FloatSlider
except Exception:
    widgets = None
    interact = None

torch.manual_seed(0)
np.random.seed(0)
torch.set_default_dtype(torch.float64)  # eigenvalues near zero need double precision to be trustworthy

### Step-2: Toy Two-Class Dataset

In [ ]:
def make_two_moons(n_samples=80, noise=0.15):
    n = n_samples // 2
    theta = np.linspace(0, np.pi, n)
    x1 = np.stack([np.cos(theta), np.sin(theta)], axis=1) + noise * np.random.randn(n, 2)
    x2 = np.stack([1 - np.cos(theta), -np.sin(theta) - 0.5], axis=1) + noise * np.random.randn(n, 2)
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n, dtype=int), np.ones(n, dtype=int)])
    idx = np.random.permutation(len(y))
    return X[idx], y[idx]

X_np, y_np = make_two_moons(80, 0.15)
X = torch.tensor(X_np)
y = torch.tensor(y_np, dtype=torch.long)

### Step-3: A Tiny MLP, Written as a Flat-Parameter Function

`torch.autograd.functional.hessian` wants a scalar function of a single tensor. Rather than fighting
`nn.Module`'s parameter tree, we write the MLP forward pass directly against dictionaries of tensors and
provide `flatten_params`/`unflatten_params` helpers so the whole network's parameters become one flat
vector $\theta \in \mathbb{R}^P$ — exactly the object we want a Hessian of.

In [ ]:
D_IN, D_OUT = 2, 2

def param_shapes(d_hidden):
    return {
        'W1': (D_IN, d_hidden), 'b1': (d_hidden,),
        'W2': (d_hidden, D_OUT), 'b2': (D_OUT,),
    }

def flatten_params(pdict, order):
    return torch.cat([pdict[k].reshape(-1) for k in order])

def unflatten_params(flat, shapes, order):
    out = {}
    idx = 0
    for k in order:
        n = int(np.prod(shapes[k]))
        out[k] = flat[idx:idx + n].reshape(shapes[k])
        idx += n
    return out

def forward(pdict, X):
    z1 = X @ pdict['W1'] + pdict['b1']
    a1 = torch.relu(z1)
    z2 = a1 @ pdict['W2'] + pdict['b2']
    return z2

def make_loss_fn(shapes, order, X, y):
    def loss_fn(flat_theta):
        pdict = unflatten_params(flat_theta, shapes, order)
        logits = forward(pdict, X)
        return torch.nn.functional.cross_entropy(logits, y)
    return loss_fn

def init_params(d_hidden, scale=0.5, seed=0):
    g = torch.Generator().manual_seed(seed)
    W1 = scale * torch.randn(D_IN, d_hidden, generator=g)
    b1 = torch.zeros(d_hidden)
    W2 = scale * torch.randn(d_hidden, D_OUT, generator=g)
    b2 = torch.zeros(D_OUT)
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

def n_params(d_hidden):
    shapes = param_shapes(d_hidden)
    return sum(int(np.prod(s)) for s in shapes.values())

for h in (8, 16, 32):
    print(f"hidden={h:3d} -> total parameters = {n_params(h)}")

### Step-4: Train Briefly (Adam) — We Want the Hessian *Near a Minimum*

In [ ]:
D_HIDDEN = 16
ORDER = ['W1', 'b1', 'W2', 'b2']

def train_and_get_theta(d_hidden=D_HIDDEN, n_steps=400, lr=0.05, seed=0):
    shapes = param_shapes(d_hidden)
    pdict0 = init_params(d_hidden, seed=seed)
    theta0 = flatten_params(pdict0, ORDER).clone().requires_grad_(True)
    loss_fn = make_loss_fn(shapes, ORDER, X, y)

    theta = theta0.detach().clone().requires_grad_(True)
    opt = torch.optim.Adam([theta], lr=lr)
    for step in range(n_steps):
        opt.zero_grad()
        loss = loss_fn(theta)
        loss.backward()
        opt.step()

    with torch.no_grad():
        pdict = unflatten_params(theta, shapes, ORDER)
        acc = (forward(pdict, X).argmax(dim=1) == y).float().mean().item()
    return theta.detach().clone(), loss_fn, loss.item(), acc

theta_eval, loss_fn, final_loss, final_acc = train_and_get_theta()
print(f"total params: {theta_eval.numel()}")
print(f"final training loss: {final_loss:.6g}")
print(f"final training accuracy: {final_acc:.3f}")

### Step-5: Compute the Exact Hessian

In [ ]:
H = torch.autograd.functional.hessian(loss_fn, theta_eval)
H = H.detach().numpy()
H = 0.5 * (H + H.T)   # symmetrize away tiny numerical asymmetry
print("Hessian shape:", H.shape)

eigvals, eigvecs = np.linalg.eigh(H)   # ascending order
print("min eigenvalue:", eigvals.min())
print("max eigenvalue:", eigvals.max())

### Step-6: Eigenvalue Spectrum — Histogram

In [ ]:
abs_eig = np.abs(eigvals)
absmax = abs_eig.max()
n_big = int(np.sum(abs_eig > 0.01 * absmax))
frac_near_zero = float(np.mean(abs_eig < 1e-3 * absmax))
print(f"# eigenvalues with |lambda| > 1% of max: {n_big} / {len(eigvals)}")
print(f"fraction of eigenvalues with |lambda| < 0.1% of max: {frac_near_zero:.3f}")

plt.figure(figsize=(7, 4))
plt.hist(eigvals, bins=60, color='C0')
plt.title(f"Hessian eigenvalue spectrum (P={len(eigvals)} params)\n"
          f"{n_big} 'large' eigenvalues (>1% of max) vs. a bulk near zero ({frac_near_zero:.1%} of all eigenvalues < 0.1% of max)")
plt.xlabel("eigenvalue"); plt.ylabel("count")
plt.grid(ls='--', lw=0.3)
plt.tight_layout()
plt.show()

**Honesty check on the qualitative shape.** With only a couple hundred parameters, our toy network is
*much* smaller than the networks Sagun et al. (2017) and Ghorbani et al. (2019) study, and the effect is
correspondingly less dramatic — we should not expect a razor-sharp two-population spectrum. What we do
see, and what's printed above, is directionally the same phenomenon: a small minority of eigenvalues
(roughly 5–15% of the total) carry the overwhelming majority of the curvature magnitude, while the rest
cluster tightly near zero. If you shrink `D_HIDDEN` a lot (e.g. down to 4), the network becomes so small
that this separation gets noisier and less clean — overparameterization relative to the (tiny) dataset is
part of what drives the near-zero bulk, so a network with barely enough capacity to fit the data doesn't
show it as cleanly. Try the interactive cell in Step-9 to see this yourself.

### Step-7: Condition Number — Why This Hurts Plain GD/SGD

In [ ]:
floor = 1e-6  # avoid dividing by (numerically) zero eigenvalues
cond = abs_eig.max() / max(abs_eig.min(), floor)
print(f"condition number (|lambda_max| / max(|lambda_min|, {floor})): {cond:.4g}")

# The largest *stable* GD step size along the sharpest direction is ~ 2 / lambda_max.
# Progress along the flattest non-trivial direction in that many steps is governed by
# roughly eta * lambda_min_nonzero -- i.e. it can be orders of magnitude slower.
eta_max_stable = 2.0 / abs_eig.max()
smallest_nonzero = abs_eig[abs_eig > floor].min() if np.any(abs_eig > floor) else floor
print(f"largest GD step size stable along the sharpest direction: eta ~ 2/lambda_max = {eta_max_stable:.4g}")
print(f"smallest 'non-degenerate' |eigenvalue| above the numerical floor: {smallest_nonzero:.4g}")
print(f"-> at that step size, curvature-driven progress along the flattest resolvable direction is "
      f"~{eta_max_stable * smallest_nonzero:.2e} per step versus ~{eta_max_stable * abs_eig.max():.2e} "
      f"along the sharpest -- a {(eta_max_stable * abs_eig.max()) / (eta_max_stable * smallest_nonzero):.3g}x gap.")

This is the concrete mechanism behind "ill-conditioned curvature hurts plain GD/SGD step-size choice":
a single global learning rate is bounded above by stability along the *sharpest* direction, but usable
progress along *flat* directions scales with that same learning rate times a much smaller curvature.
Methods that rescale each direction by (an estimate of) its own curvature — Newton's method exactly,
Natural Gradient / K-FAC / EKFAC approximately — sidestep this by taking large steps in flat directions
and small, safe steps in sharp ones, in the *same* update.

### Step-8: Flat vs. Sharp Directions — 1D Loss Curves Along Eigenvectors

In [ ]:
sharp_idx = np.argmax(np.abs(eigvals))
flat_idx = np.argmin(np.abs(eigvals))
sharp_vec = eigvecs[:, sharp_idx]
flat_vec = eigvecs[:, flat_idx]
print(f"sharp eigenvalue: {eigvals[sharp_idx]:.6g}   flat eigenvalue: {eigvals[flat_idx]:.3g} (numerically ~0)")

def loss_along_direction(theta_center, direction, ts):
    direction_t = torch.tensor(direction)
    losses = []
    for t in ts:
        theta_t = theta_center + t * direction_t
        with torch.no_grad():
            losses.append(loss_fn(theta_t).item())
    return np.array(losses)

ts = np.linspace(-2.0, 2.0, 61)
loss_sharp = loss_along_direction(theta_eval, sharp_vec, ts)
loss_flat = loss_along_direction(theta_eval, flat_vec, ts)

plt.figure(figsize=(7, 4))
plt.plot(ts, loss_sharp, label=f"sharp direction ($\\lambda$={eigvals[sharp_idx]:.3g})")
plt.plot(ts, loss_flat, label=f"flat direction ($\\lambda$={eigvals[flat_idx]:.1e})")
plt.xlabel("t  (perturbation size along eigenvector)")
plt.ylabel(r"loss($\theta^* + t \cdot$ eigvec)")
plt.title("1D loss curves along the sharpest vs. flattest eigenvector")
plt.legend(); plt.grid(ls='--', lw=0.3)
plt.tight_layout()
plt.show()

print(f"loss range along sharp direction: [{loss_sharp.min():.4g}, {loss_sharp.max():.4g}]")
print(f"loss range along flat direction:  [{loss_flat.min():.4g}, {loss_flat.max():.4g}]")

As expected from the eigenvalues themselves: perturbing along the **sharp** eigenvector produces a
narrow, rapidly-rising parabola (loss changes by orders of magnitude within $|t| \le 2$), while
perturbing along the **flat** eigenvector barely moves the loss at all — visually indistinguishable from a
flat line at this scale. This is the geometric picture behind "flat minima" language in the
generalization literature: some directions in parameter space are essentially unconstrained by the
training loss near a minimum.

### Step-9: Interactive Exploration — Hidden Width vs. Spectrum Shape

In [ ]:
if widgets is None:
    display(Markdown("**ipywidgets not installed — run the static cells above instead.**"))
else:
    def interactive_hessian(d_hidden=16, n_steps=400, eig_rank_from_top=0):
        theta_i, loss_fn_i, loss_i, acc_i = train_and_get_theta(d_hidden=d_hidden, n_steps=n_steps)
        H_i = torch.autograd.functional.hessian(loss_fn_i, theta_i).detach().numpy()
        H_i = 0.5 * (H_i + H_i.T)
        eigvals_i, eigvecs_i = np.linalg.eigh(H_i)
        abs_eig_i = np.abs(eigvals_i)
        absmax_i = abs_eig_i.max()
        n_big_i = int(np.sum(abs_eig_i > 0.01 * absmax_i))
        cond_i = absmax_i / max(abs_eig_i.min(), 1e-6)

        # rank 0 = sharpest, rank -1 = flattest, by |eigenvalue|
        order_by_abs = np.argsort(-abs_eig_i)
        chosen_idx = order_by_abs[min(eig_rank_from_top, len(order_by_abs) - 1)]
        chosen_vec = eigvecs_i[:, chosen_idx]

        display(Markdown(
            f"**hidden={d_hidden}, params={theta_i.numel()}, train steps={n_steps}** &nbsp;|&nbsp; "
            f"final loss={loss_i:.4g}, acc={acc_i:.3f} &nbsp;|&nbsp; "
            f"condition number={cond_i:.4g} &nbsp;|&nbsp; "
            f"{n_big_i}/{len(eigvals_i)} eigenvalues > 1% of max"
        ))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(eigvals_i, bins=50)
        axes[0].set_title("Eigenvalue spectrum")
        axes[0].set_xlabel("eigenvalue"); axes[0].set_ylabel("count")
        axes[0].grid(ls='--', lw=0.3)

        ts = np.linspace(-2.0, 2.0, 61)
        direction_t = torch.tensor(chosen_vec)
        losses = []
        for t in ts:
            with torch.no_grad():
                losses.append(loss_fn_i(theta_i + t * direction_t).item())
        axes[1].plot(ts, losses)
        axes[1].set_title(f"Loss along eigenvector rank {eig_rank_from_top} "
                           f"($\\lambda$={eigvals_i[chosen_idx]:.3g})")
        axes[1].set_xlabel("t"); axes[1].set_ylabel("loss")
        axes[1].grid(ls='--', lw=0.3)
        plt.tight_layout()
        plt.show()

    interact(
        interactive_hessian,
        d_hidden=IntSlider(value=16, min=4, max=48, step=4, description='hidden width'),
        n_steps=IntSlider(value=400, min=20, max=800, step=20, description='train steps'),
        eig_rank_from_top=IntSlider(value=0, min=0, max=40, step=1, description='eigvec rank'),
    )

## ✅ Practical Notes & Takeaways

- **The Hessian spectrum near a trained minimum is not uniform.** Even in a network with only ~80
  parameters, a small minority of directions carry the overwhelming majority of curvature, and the rest
  are nearly flat. This gets qualitatively sharper (more extreme) as models get larger — see Sagun et al.
  (2017), Ghorbani et al. (2019) for the large-scale empirical picture.
- **Condition number is the concrete link to optimizer design.** A large ratio between the sharpest and
  flattest (non-degenerate) curvature directions means a single global learning rate is a bad compromise:
  stable-but-slow, or fast-but-unstable, never both. This is the entire motivation for preconditioning
  (Newton, Natural Gradient, K-FAC, EKFAC — see the other notebooks in this repo).
- **Exact Hessians don't scale.** Forming and eigendecomposing a $P \times P$ matrix is $O(P^2)$ memory
  and $O(P^3)$ compute — fine for ~200 parameters, utterly infeasible for a modern network. This is
  exactly why Kronecker-factored (K-FAC/EKFAC), diagonal (Adam-style), or Hessian-vector-product-based
  (Lanczos, conjugate gradient) approximations exist: they trade exactness for tractable cost while trying
  to preserve the parts of the curvature that matter most for optimization.
- **What we did not do:** compute Hessians for anything beyond a toy scale, use stochastic/mini-batch
  Hessian estimates (ours is a full-batch Hessian on the tiny toy dataset), or study how the spectrum
  evolves *during* training rather than at a single snapshot (the interactive cell lets you approximate
  this by varying `n_steps`).

## 🧾 Summary

- We computed the **exact** Hessian of a tiny MLP's loss via `torch.autograd.functional.hessian` and
  eigendecomposed it directly with `numpy.linalg.eigvalsh`/`eigh`.
- The spectrum showed the qualitative shape reported for real (much larger) networks near a minimum: a
  small number of large eigenvalues plus a bulk clustered near zero — noisier and less extreme at our toy
  scale, as discussed honestly in Step-6.
- The condition number quantifies exactly why a single learning rate struggles under this kind of
  curvature, connecting directly to the motivation for the curvature-aware optimizers covered elsewhere
  in this repo.
- Perturbing along the top/bottom eigenvectors made the flat-vs-sharp distinction visually concrete: a
  narrow parabola along the sharp direction, an essentially flat line along the flat direction.

**References**
- Sagun, L., Evci, U., Güney, V. U., Dauphin, Y., & Bottou, L. (2017). *Empirical Analysis of the
  Hessian of Over-Parametrized Neural Networks.*
- Ghorbani, B., Krishnan, S., & Xiao, Y. (2019). *An Investigation into Neural Net Optimization via
  Hessian Eigenvalue Density.*
- Martens, J., & Grosse, R. (2015). *Optimizing Neural Networks with Kronecker-Factored Approximate
  Curvature.*